In [5]:
from pathlib import Path
import os
import sys
import time
import tracemalloc

import torch

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("project_root:", project_root)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda_device:", torch.cuda.get_device_name(0))

project_root: c:\Users\Zhai-Bin Cui\Desktop\GPTScratch\mini-train-sys
torch: 2.5.1+cu121
cuda_available: True
cuda_device: NVIDIA GeForce RTX 3050 Laptop GPU


In [6]:
from minitrain.data.dataloader import build_training_dataloader
from minitrain.model import MiniTransformer, ModelConfig
from minitrain.runtime.config import experiment_config_from_dict
from minitrain.runtime.device import get_default_device
from minitrain.runtime.factory import build_ops_backend, build_parallel_strategy
from minitrain.train.optim import build_optimizer
from minitrain.train.trainer import Trainer
from minitrain.utils.seed import seed_everything

In [7]:
payload = {
    "run": {"name": "notebook_torch_backend", "seed": 1337},
    "backend": {"ops": "torch", "parallel": "single"},
    "train": {
        "batch_size": 4,
        "lr": 3e-4,
        "weight_decay": 0.1,
        "max_steps": 5,
        "log_interval": 1,
        "use_fused_loss": False,
    },
    "data": {"source": "random", "num_tokens": 8192, "shuffle": True},
    "logging": {"console": False, "tensorboard": False},
    "model": {
        "vocab_size": 2048,
        "seq_len": 64,
        "n_layers": 2,
        "n_heads": 4,
        "hidden_size": 128,
        "intermediate_size": 256,
        "dropout": 0.0,
    },
}

cfg = experiment_config_from_dict(payload)
model_cfg = ModelConfig(**cfg.model)
cfg, model_cfg

(ExperimentConfig(run=RunConfig(name='notebook_torch_backend', seed=1337), backend=BackendConfig(ops='torch', parallel='single'), train=TrainConfig(batch_size=4, lr=0.0003, weight_decay=0.1, max_steps=5, log_interval=1, use_fused_loss=False), data=DataConfig(source='random', path=None, num_tokens=8192, shuffle=True), logging=LoggingConfig(console=False, tensorboard=False, log_dir='runs', flush_secs=10), distributed={}, model={'vocab_size': 2048, 'seq_len': 64, 'n_layers': 2, 'n_heads': 4, 'hidden_size': 128, 'intermediate_size': 256, 'dropout': 0.0}),
 ModelConfig(vocab_size=2048, seq_len=64, n_layers=2, n_heads=4, hidden_size=128, intermediate_size=256, norm_eps=1e-05, rope_theta=10000.0, dropout=0.0, tie_word_embeddings=True))

In [8]:
seed_everything(cfg.run.seed)

device = get_default_device()
if device.type == "cuda":
    torch.cuda.set_device(device)

ops = build_ops_backend(cfg.backend)
model = MiniTransformer(model_cfg, ops).to(device)
dataloader = build_training_dataloader(
    cfg.data,
    seq_len=model_cfg.seq_len,
    batch_size=cfg.train.batch_size,
    vocab_size=model_cfg.vocab_size,
    seed=cfg.run.seed,
)

strategy = build_parallel_strategy(cfg)
strategy.setup()
model = strategy.wrap_model(model)
optimizer = build_optimizer(model, lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
trainer = Trainer(model, optimizer, device=device, use_fused_loss=cfg.train.use_fused_loss)

param_count = sum(p.numel() for p in model.parameters())
print({
    "device": str(device),
    "ops": ops.name,
    "parallel": strategy.name,
    "params": param_count,
    "batch_size": cfg.train.batch_size,
    "seq_len": model_cfg.seq_len,
})

{'device': 'cuda:0', 'ops': 'torch', 'parallel': 'single', 'params': 590464, 'batch_size': 4, 'seq_len': 64}


In [9]:
batch = next(iter(dataloader))
print("input_ids:", batch["input_ids"].shape, batch["input_ids"].dtype)
print("targets:  ", batch["targets"].shape, batch["targets"].dtype)
print("shift check:", torch.equal(batch["input_ids"][:, 1:], batch["targets"][:, :-1]))

input_ids: torch.Size([4, 64]) torch.int64
targets:   torch.Size([4, 64]) torch.int64
shift check: True


In [26]:
torch.cuda.empty_cache()
torch.cuda.memory_allocated(device) / 1024**2

23.046875

In [27]:
def repeat_dataloader(loader):
    while True:
        yield from loader


def memory_metrics_mb(device):
    if device.type == "cuda":
        return {
            "gpu_allocated_mb": round(torch.cuda.memory_allocated(device) / 1024**2, 2),
            "gpu_reserved_mb": round(torch.cuda.memory_reserved(device) / 1024**2, 2),
            "gpu_peak_allocated_mb": round(torch.cuda.max_memory_allocated(device) / 1024**2, 2),
        }
    _, peak_bytes = tracemalloc.get_traced_memory()
    return {"host_peak_memory_mb": round(peak_bytes / 1024**2, 2)}


history = []
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)
else:
    tracemalloc.start()

strategy.barrier()
start_time = time.perf_counter()
last_log_time = start_time
last_log_tokens = 0

try:
    trainer.state.step = 0
    for batch in repeat_dataloader(dataloader):
        loss = trainer.train_step(batch)
        should_log = trainer.state.step == 1 or trainer.state.step % cfg.train.log_interval == 0
        should_stop = trainer.state.step >= cfg.train.max_steps

        if should_log or should_stop:
            strategy.barrier()
            now = time.perf_counter()
            interval_tokens = trainer.state.tokens_seen - last_log_tokens
            event = {
                "step": trainer.state.step,
                "loss": round(float(loss), 6),
                "tokens_seen": trainer.state.tokens_seen,
                "tokens_per_sec": round(interval_tokens / max(now - last_log_time, 1e-12), 2),
                "avg_tokens_per_sec": round(trainer.state.tokens_seen / max(now - start_time, 1e-12), 2),
            }
            event.update(memory_metrics_mb(device))
            history.append(event)
            print(event)
            last_log_time = now
            last_log_tokens = trainer.state.tokens_seen

        if should_stop:
            break
finally:
    if device.type != "cuda" and tracemalloc.is_tracing():
        tracemalloc.stop()

# estimated random guess loss
print(torch.math.log(cfg.model['vocab_size']))

{'step': 1, 'loss': 7.63259, 'tokens_seen': 5120, 'tokens_per_sec': 31533.5, 'avg_tokens_per_sec': 31533.5, 'gpu_allocated_mb': 23.05, 'gpu_reserved_mb': 54.0, 'gpu_peak_allocated_mb': 36.44}
{'step': 2, 'loss': 7.580218, 'tokens_seen': 5376, 'tokens_per_sec': 17612.78, 'avg_tokens_per_sec': 30389.72, 'gpu_allocated_mb': 23.05, 'gpu_reserved_mb': 54.0, 'gpu_peak_allocated_mb': 36.44}
{'step': 3, 'loss': 7.632232, 'tokens_seen': 5632, 'tokens_per_sec': 20289.12, 'avg_tokens_per_sec': 29717.26, 'gpu_allocated_mb': 23.05, 'gpu_reserved_mb': 54.0, 'gpu_peak_allocated_mb': 36.44}
{'step': 4, 'loss': 7.59703, 'tokens_seen': 5888, 'tokens_per_sec': 22734.34, 'avg_tokens_per_sec': 29325.63, 'gpu_allocated_mb': 23.05, 'gpu_reserved_mb': 54.0, 'gpu_peak_allocated_mb': 36.44}
{'step': 5, 'loss': 7.562489, 'tokens_seen': 6144, 'tokens_per_sec': 19668.4, 'avg_tokens_per_sec': 28737.7, 'gpu_allocated_mb': 23.05, 'gpu_reserved_mb': 54.0, 'gpu_peak_allocated_mb': 36.44}
7.6246189861593985


In [28]:
print("final_step:", trainer.state.step)
print("tokens_seen:", trainer.state.tokens_seen)
print("last_loss:", history[-1]["loss"] if history else None)

strategy.teardown()

final_step: 5
tokens_seen: 6144
last_loss: 7.562489


In [33]:
import gc
import subprocess

for name in [
    "loss", "logits", "batch",
    "trainer", "optimizer", "model", "dataloader", "strategy",
]:
    globals().pop(name, None)
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

cmd = [
    sys.executable,
    "scripts/train.py",
    "--config",
    "configs/train_smoke_gpu.yaml",
    "--model-config",
    "configs/model_tiny_smoke.yaml",
    "--smoke-steps",
    "1",
    "--device",
    "cuda" if torch.cuda.is_available() else "cpu",
]
print(" ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

d:\anaconda\envs\llm\python.exe scripts/train.py --config configs/train_smoke_gpu.yaml --model-config configs/model_tiny_smoke.yaml --smoke-steps 1 --device cuda
{'event': 'init', 'run': 'single_torch_smoke_gpu', 'device': 'cuda:0', 'ops': 'torch', 'parallel': 'single', 'world_size': 1, 'params': 590464, 'data_source': 'random', 'batch_size': 1, 'seq_len': 64, 'tensorboard_dir': 'disabled'}
{'event': 'train', 'step': 1, 'loss': 7.636544, 'tokens_seen': 64, 'tokens_per_sec': 66.84, 'avg_tokens_per_sec': 66.84, 'gpu_memory_allocated_mb': 23.05, 'gpu_memory_reserved_mb': 30.0, 'gpu_peak_memory_allocated_mb': 25.8}

